# Nettoyage et préparation de la base — Portefeuille de crédit

**Projet :** analyse d'un portefeuille fictif de 44 991 emprunteurs pour dashboards Tableau Public.

Ce notebook consolide en un seul fichier l'ensemble des traitements appliqués à la base :

1. Chargement de la base brute (`loan_data.csv`)
2. Audit qualité
3. Nettoyage (types, noms de colonnes, valeurs manquantes, doublons)
4. Contrôles de cohérence (bornes réalistes)
5. Traitement des outliers (winsorisation 1%/99%)
6. Feature engineering (tranches, ratios, variable cible)
7. Renommage des colonnes en français et export intermédiaire
8. Re-segmentation de `tranche_revenu` avec quantiles asymétriques
9. Export final de `base_final.csv` (à importer dans Tableau)
10. Exports complémentaires pour Tableau (matrice de corrélations, taux de défaut par endettement)

**Sortie principale :** `base_final.csv` — 44 991 lignes × 22 colonnes


## 1) Imports et configuration

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Chemins
INPUT_PATH        = "loan_data.csv"
INTERMEDIATE_CSV  = "clean_bank_loans_fr_final.csv"
FINAL_CSV         = "base_final.csv"
CORR_CSV          = "correlation_matrix.csv"
DEFAULT_RATE_CSV  = "taux_defaut_endettement.csv"


## 2) Chargement des données

In [ ]:
df = pd.read_csv(INPUT_PATH, sep=";")

print("Shape brut :", df.shape)
df.head()


## 3) Audit rapide

Objectif : comprendre la qualité des données avant nettoyage (missing, doublons, modalités catégorielles).


In [ ]:
def missing_report(data: pd.DataFrame) -> pd.DataFrame:
    miss = data.isna().sum()
    pct = (miss / len(data) * 100).round(2)
    rep = pd.DataFrame({"missing": miss, "missing_pct": pct})
    return rep[rep["missing"] > 0].sort_values("missing_pct", ascending=False)

print("Valeurs manquantes :")
print(missing_report(df))

print("\nDoublons :", df.duplicated().sum())

cat_cols = ["person_gender", "person_education", "person_home_ownership",
            "loan_intent", "previous_loan_defaults_on_file", "loan_status"]
for c in cat_cols:
    if c in df.columns:
        print(f"\n{c} — modalités :", df[c].dropna().unique()[:20])


## 4) Nettoyage

- Standardisation des noms de colonnes
- Harmonisation des catégories (lowercase, trim)
- Conversion robuste des colonnes numériques
- Mise au format binaire de `loan_status` et `previous_loan_defaults_on_file`
- Suppression des doublons


In [ ]:
# 4.1) Noms de colonnes en snake_case
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 4.2) Nettoyage des colonnes catégorielles
def clean_text_series(s: pd.Series) -> pd.Series:
    return (s.astype("string").str.strip().str.lower().replace({"": pd.NA}))

for col in ["person_gender", "person_education", "person_home_ownership",
            "loan_intent", "previous_loan_defaults_on_file"]:
    if col in df.columns:
        df[col] = clean_text_series(df[col])

# 4.3) Conversion types numériques (robuste aux valeurs invalides)
num_cols = ["person_age", "person_income", "person_emp_exp", "loan_amnt",
            "loan_int_rate", "loan_percent_income",
            "cb_person_cred_hist_length", "credit_score"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 4.4) loan_status -> binaire 0/1
if "loan_status" in df.columns:
    if pd.api.types.is_numeric_dtype(df["loan_status"]):
        df["loan_status"] = pd.to_numeric(df["loan_status"], errors="coerce")
    else:
        mapping = {"0": 0, "1": 1, "no": 0, "yes": 1,
                   "paid": 0, "non_default": 0, "fully_paid": 0,
                   "default": 1, "charged_off": 1}
        df["loan_status"] = clean_text_series(df["loan_status"]).map(mapping)

# 4.5) previous_loan_defaults_on_file -> 0/1
if "previous_loan_defaults_on_file" in df.columns:
    mapping_prev = {"no": 0, "yes": 1, "0": 0, "1": 1, "false": 0, "true": 1}
    df["previous_loan_defaults_on_file"] = df["previous_loan_defaults_on_file"].map(mapping_prev)

# 4.6) Suppression des doublons
df = df.drop_duplicates()

print("Shape après nettoyage :", df.shape)


## 5) Gestion des valeurs manquantes

Stratégie :
- cible `loan_status` manquante → suppression de la ligne
- numériques → imputation par la médiane (robuste aux outliers)
- catégorielles → remplacement par `"unknown"`


In [ ]:
# Suppression des lignes sans cible
if "loan_status" in df.columns:
    df = df.dropna(subset=["loan_status"])

# Imputation médiane pour les numériques
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Catégorielles -> unknown
for col in ["person_gender", "person_education", "person_home_ownership", "loan_intent"]:
    if col in df.columns:
        df[col] = df[col].fillna("unknown")

print("Missing restants :", df.isna().sum().sum())


## 6) Contrôles de cohérence

Filtrage sur bornes réalistes pour éviter que des erreurs de saisie ne faussent les analyses :
- âge : 18–80 ans
- score de crédit : 300–900
- taux d'intérêt : 0–60 %
- `loan_percent_income` : ratio entre 0 et 2 (conversion auto si exprimé en %)
- pas de valeurs négatives sur revenus, montants, expérience


In [ ]:
if "person_age" in df.columns:
    df = df[(df["person_age"] >= 18) & (df["person_age"] <= 80)]

if "credit_score" in df.columns:
    df = df[(df["credit_score"] >= 300) & (df["credit_score"] <= 900)]

if "loan_int_rate" in df.columns:
    df = df[(df["loan_int_rate"] >= 0) & (df["loan_int_rate"] <= 60)]

# loan_percent_income : détection automatique % ou ratio
if "loan_percent_income" in df.columns:
    if df["loan_percent_income"].median() > 2:
        df["loan_percent_income"] = df["loan_percent_income"] / 100.0
    df = df[(df["loan_percent_income"] >= 0) & (df["loan_percent_income"] <= 2)]

# Pas de valeurs négatives
for col in ["person_income", "loan_amnt", "person_emp_exp", "cb_person_cred_hist_length"]:
    if col in df.columns:
        df = df[df[col] >= 0]

print("Shape après validité :", df.shape)


## 7) Traitement des outliers

Winsorisation 1%/99% sur les variables sensibles (revenu, montant, taux, score) pour stabiliser les visualisations Tableau.


In [ ]:
def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99) -> pd.Series:
    lo, hi = s.quantile(lower_q), s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

for col in ["person_income", "loan_amnt", "loan_int_rate", "credit_score"]:
    if col in df.columns:
        df[col] = winsorize_series(df[col], 0.01, 0.99)


## 8) Feature engineering

Création des variables utilisées dans les dashboards Tableau :

| Variable | Description |
|---|---|
| `default` | Cible binaire 0/1 |
| `age_group` | Tranches d'âge |
| `income_band` | Quintiles de revenu (labels FR) |
| `credit_band` | Quintiles de score de crédit (labels FR) |
| `loan_to_income` | Ratio prêt / revenu recalculé |
| `ratio_gap_abs` | Écart absolu entre ratio recalculé et ratio d'origine |
| `emp_exp_band` | Tranches d'expérience professionnelle |


In [ ]:
# Variable cible binaire
df["default"] = df["loan_status"].astype(int)

# Tranches d'âge
df["age_group"] = pd.cut(
    df["person_age"],
    bins=[18, 25, 35, 45, 60, 80],
    labels=["18-25", "25-35", "35-45", "45-60", "60-80"],
    include_lowest=True
)

# Quintiles de revenu — labels FR
df["income_band"] = pd.qcut(
    df["person_income"],
    q=5,
    labels=["Très faible revenu", "Faible revenu", "Revenu moyen",
            "Revenu élevé", "Très haut revenu"]
)

# Quintiles de score de crédit — labels FR
df["credit_band"] = pd.qcut(
    df["credit_score"],
    q=5,
    labels=["Très faible score", "Faible score", "Score moyen",
            "Bon score", "Très bon score"]
)

# Ratio prêt/revenu recalculé
df["loan_to_income"] = df["loan_amnt"] / df["person_income"].replace({0: np.nan})
df["loan_to_income"] = df["loan_to_income"].fillna(df["loan_to_income"].median())

# Écart entre ratio recalculé et ratio d'origine (détection anomalies)
df["ratio_gap_abs"] = (df["loan_to_income"] - df["loan_percent_income"]).abs()

# Tranches d'expérience professionnelle
df["emp_exp_band"] = pd.cut(
    df["person_emp_exp"],
    bins=[0, 1, 3, 5, 10, 50],
    labels=["0-1", "1-3", "3-5", "5-10", "10+"],
    include_lowest=True
)


## 9) Renommage en français + ID de ligne + export intermédiaire

Sélection et ordonnancement des colonnes, renommage en français pour Tableau, ajout d'un `id_ligne` unique.


In [ ]:
# Sélection et ordre des colonnes finales
final_cols = [
    "person_age", "age_group",
    "person_gender", "person_education", "person_home_ownership",
    "person_income", "income_band",
    "person_emp_exp", "emp_exp_band",
    "loan_amnt", "loan_intent", "loan_int_rate",
    "loan_percent_income", "loan_to_income", "ratio_gap_abs",
    "cb_person_cred_hist_length",
    "credit_score", "credit_band",
    "previous_loan_defaults_on_file",
    "loan_status", "default",
]
final_cols = [c for c in final_cols if c in df.columns]
df_clean = df[final_cols].copy()

# Renommage en français
rename_dict = {
    "person_age":                     "age",
    "person_gender":                  "genre",
    "person_education":               "niveau_education",
    "person_income":                  "revenu_annuel",
    "person_emp_exp":                 "experience_professionnelle",
    "person_home_ownership":          "statut_logement",
    "loan_amnt":                      "montant_pret",
    "loan_intent":                    "objectif_pret",
    "loan_int_rate":                  "taux_interet",
    "loan_percent_income":            "ratio_endettement_calcule",
    "cb_person_cred_hist_length":     "anciennete_credit",
    "credit_score":                   "score_credit",
    "previous_loan_defaults_on_file": "antecedent_defaut",
    "loan_status":                    "statut_pret",
    "default":                        "defaut",
    "age_group":                      "tranche_age",
    "income_band":                    "tranche_revenu",
    "ratio_gap_abs":                  "ecart_ratio_endettement",
    "credit_band":                    "categorie_score_credit",
    "loan_to_income":                 "ratio_pret_revenu_calcule",
    "emp_exp_band":                   "tranche_experience",
}
df_clean_fr = df_clean.rename(columns=rename_dict)

# Ajout d'un identifiant unique de ligne
df_clean_fr.insert(0, "id_ligne", range(1, len(df_clean_fr) + 1))

# Export intermédiaire
df_clean_fr.to_csv(INTERMEDIATE_CSV, index=False, sep=";", encoding="latin-1")

print("Shape nettoyée :", df_clean_fr.shape)
print("Colonnes :", df_clean_fr.columns.tolist())
print(f"\nExport : {INTERMEDIATE_CSV}")


## 10) Re-segmentation de `tranche_revenu` — quintiles asymétriques

Les quintiles égaux (20/20/20/20/20) donnent une distribution trop uniforme pour la visualisation. On redéfinit les bornes sur des percentiles asymétriques (35/30/20/10/5) pour refléter la distribution réelle des revenus — la plupart des emprunteurs ont des revenus faibles ou moyens, les très hauts revenus sont rares.


In [ ]:
# On recharge la base pour isoler cette étape (reproductibilité)
df = pd.read_csv(INTERMEDIATE_CSV, sep=";", encoding="latin-1")

# Bornes asymétriques
p = df["revenu_annuel"].quantile([0, 0.35, 0.65, 0.85, 0.95, 1])
pv = p.values

df["tranche_revenu"] = pd.cut(
    df["revenu_annuel"],
    bins=pv,
    labels=["Très faible revenu", "Faible revenu", "Revenu moyen",
            "Revenu élevé", "Très haut revenu"],
    include_lowest=True
)

print("--- Nouvelle répartition ---")
print(df["tranche_revenu"].value_counts())
print("\n--- Proportions ---")
print(df["tranche_revenu"].value_counts(normalize=True).round(3))

# Sauvegarde finale
df.to_csv(FINAL_CSV, index=False, encoding="utf-8")
print(f"\nExport final : {FINAL_CSV}")


In [ ]:
# Vérification : bornes min/max par tranche
resume_tranches = df.groupby("tranche_revenu", observed=True)["revenu_annuel"].agg(
    borne_min="min",
    borne_max="max",
    effectif="count"
).reset_index()

print(resume_tranches)


## 11) Export de la matrice de corrélations pour Tableau

Tableau Public ne calcule pas nativement les corrélations entre colonnes. On exporte une version "longue" (long format) avec 3 colonnes : `variable_1`, `variable_2`, `correlation`. Elle sera importée dans Tableau pour générer la heatmap de corrélations.


In [ ]:
cols = ["revenu_annuel", "montant_pret", "score_credit", "taux_interet"]
corr = df[cols].corr().round(2)

# Passage en format long pour Tableau
corr_long = corr.reset_index().melt(id_vars="index")
corr_long.columns = ["variable_1", "variable_2", "correlation"]
corr_long.to_csv(CORR_CSV, index=False)

print(f"Export : {CORR_CSV}")
print(corr_long.head(10))


## 12) Taux de défaut par catégorie d'endettement

Analyse complémentaire : on calcule le taux de défaut par tranche de `ratio_endettement_calcule`. Ces résultats alimentent la feuille Tableau sur le risque, et démontrent que le ratio d'endettement est un **bien meilleur prédicteur** du défaut que le score de crédit.


In [ ]:
bins   = [0, 0.10, 0.20, 0.30, 0.40, 1.0]
labels = ["< 10%", "10-20%", "20-30%", "30-40%", "> 40%"]

df["cat_endettement"] = pd.cut(df["ratio_endettement_calcule"],
                                bins=bins, labels=labels)

resultat = df.groupby("cat_endettement", observed=True)["defaut"].agg(
    Effectif="count",
    Defauts="sum"
)
resultat["Taux_defaut_pct"] = (resultat["Defauts"] / resultat["Effectif"] * 100).round(1)

print(resultat)
resultat.to_csv(DEFAULT_RATE_CSV)
print(f"\nExport : {DEFAULT_RATE_CSV}")


## 13) Contrôles finaux

In [ ]:
print("Dimensions de la base finale :", df.shape)
print("\nValeurs manquantes restantes :")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nTaux de défaut global :", round(df["defaut"].mean(), 4))

print("\nColonnes finales :")
print(df.columns.tolist())
